# <center> Автоэнкодеры (AE) и Вариационные автоэнкодеры (VAE) в RecSys </center> 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import bottleneck as bn
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import random
import pandas as pd
import seaborn as sn
import shutil
import sys
import torch
torch.set_num_threads(1)

sn.set()

from scipy import sparse

In [ ]:
torch.__version__

Отступление в виде статьи, которое даст обоснование, почему мы будем рассматривать именно Mult-VAE и Mult-DAE для RecSys из большого множества методов. 

**Ferrari Dacrema, M., Cremonesi, P. and Jannach, D., 2019, September. Are we really making much progress? A worrying analysis of recent neural recommendation approaches. In Proceedings of the 13th ACM conference on recommender systems (pp. 101-109).**

Статья: https://arxiv.org/pdf/1907.06902.pdf 

Код: https://github.com/MaurizioFD/RecSys2019_DeepLearning_Evaluation?utm_source=catalyzex.com 


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything()

Mult-VAE - оказалась чуть ли не единственной моделью, которая смогла подтвердить свои результаты при воспроизведении и не проиграть простым бейзлайнам. 

Давайте вспомним, что такое автоэнкодеры, и обсудим, почему они применимы и эффективны в RecSys?

Теперь к RecSys адаптациям и особенностям.

**Mult-VAE** - обобщение вариационного автоэнкодера для коллаборативной фильтрации. 

**Liang, D., Krishnan, R.G., Hoffman, M.D. and Jebara, T., 2018, April. Variational autoencoders for collaborative filtering. In Proceedings of the 2018 world wide web conference (pp. 689-698).**

Статья: https://arxiv.org/pdf/1802.05814.pdf 

Формализация из статьи:



$u \in \{1,\dots,U\}$ - пользователи, 

$i \in \{1,\dots,I\}$ - объекты (айтемы),

$\mathbf{X} \in \mathbb{N}^{U\times I}$ - матрица интеракций из кликов в бинарном виде. 

$$\mathbf{x}_u =[X_{u1},\dots,X_{uI}]^\top \in \mathbb{N}^I$$ - вектор по айтемам с числом кликов пользователя по каждому из них. 

Функция потерь для Mult-VAE для каждого пользователя $u$:

$$\mathcal{L}_u(\theta, \phi) = \mathbb{E}_{q_\phi(z_u | x_u)}[\log p_\theta(x_u | z_u)] - \beta \cdot KL(q_\phi(z_u | x_u) \| p(z_u))$$

где $q_\phi$ - аппроксимирующее вариационное распределение (inference модель), $\beta$ - доп. параметр имитации отжига. 

Целевая функция для всего датасета - усреднение по пользователям. 


Генеративный процесс: Для каждого пользователя $u$, модель сэмплирует $K$-размерный латентный вектор  $\mathbf{z}_u$ из стандартного нормального распределения. Вектор $\mathbf{z}_u$ преобразуется нелинейной функцией $f_\theta (\cdot) \in \mathbb{R}^I$, чтобы получить вероятностное распределение по всем $I$ айтемам $\pi (\mathbf{z}_u)$ на основе имевшегося $\mathbf{x}_u$.
$$
\mathbf{z}_u \sim \mathcal{N}(0, \mathbf{I}_K),  \pi(\mathbf{z}_u) \propto \exp\{f_\theta (\mathbf{z}_u\},\\
\mathbf{x}_u \sim \mathrm{Mult}(N_u, \pi(\mathbf{z}_u))
$$

Целевая функция для Multi-DAE для каждого пользователя $u$:
$$
\mathcal{L}_u(\theta, \phi) = \log p_\theta(\mathbf{x}_u | g_\phi(\mathbf{x}_u))
$$
где $g_\phi(\cdot)$ - нелинейная "encoder" функция.


Возьмем датасет от KION по фильмам.

В статье делалем следующий препроцессинг:
- Модель авторов обучается на бинарных данных, поэтому нужно задать порог для оценок, выше которого считаем за 1.
- Разбить пользователей на training/validation/test. Обучаться будем на полной истории кликов пользователей из train, а предсказываться на новых пользователях, что часто приближено к реальным условиям индустриальных датасетов, хотя тоже есть свои ремарки. 
- Среди отложенных пользователей, 80% кликов оставить на train и 20% - на подсчет метрик качества. 

Обратимся к датасету KION:

https://github.com/irsafilo/KION_DATASET

In [ ]:
DATA_DIR = ''

raw_data = pd.read_csv('../../interactions.csv')
raw_data.head()

In [ ]:
sampled_users = np.random.choice(raw_data.user_id.unique(), size=3000)
raw_data = raw_data.loc[raw_data.user_id.isin(sampled_users)].copy()

In [ ]:
raw_data.rename({'user_id':'userId','item_id':'movieId','last_watch_dt':'timestamp'}, axis=1, inplace=True)

In [ ]:
def get_count(tp, id):
    playcount_groupbyid = tp[[id]].groupby(id, as_index=False)
    count = playcount_groupbyid.size()
    return count

In [ ]:
def filter_triplets(tp, min_uc=0, min_sc=0): 
    if min_sc > 0:
        itemcount = get_count(tp, 'movieId')
        tp = tp[tp['movieId'].isin(itemcount.index[itemcount >= min_sc])]
    
    if min_uc > 0:
        usercount = get_count(tp, 'userId')
        tp = tp[tp['userId'].isin(usercount.index[usercount >= min_uc])]
    
    usercount, itemcount = get_count(tp, 'userId'), get_count(tp, 'movieId') 
    return tp, usercount, itemcount

In [ ]:
raw_data, user_activity, item_popularity = filter_triplets(raw_data)

Мы запустим без фильтров, но попробуйте с ними, чтобы понять, повлияет ли это на качество в лучшую сторону.

In [ ]:
sparsity = raw_data.shape[0] / (user_activity.shape[0] * item_popularity.shape[0])

print("В датасете %d интеракций от %d пользователей и всего по %d фильмов (sparsity: %.3f%%)" % 
      (raw_data.shape[0], user_activity.shape[0], item_popularity.shape[0], sparsity * 100))

Сделаем train/test/val сплит

In [ ]:
unique_uid = raw_data.userId.unique()

np.random.seed(42)
idx_perm = np.random.permutation(unique_uid.size)
unique_uid = unique_uid[idx_perm]
n_users = unique_uid.size
n_heldout_users = 200

tr_users = unique_uid[:(n_users - n_heldout_users * 2)]
vd_users = unique_uid[(n_users - n_heldout_users * 2): (n_users - n_heldout_users)]
te_users = unique_uid[(n_users - n_heldout_users):]

train_plays = raw_data.loc[raw_data['userId'].isin(tr_users)]
unique_sid = pd.unique(train_plays['movieId'])

In [ ]:
len(tr_users), len(vd_users), len(te_users)

И encoder каким-нибудь другим способом, которым еще не делали, для разнообраия). Например, простой словарь.

In [ ]:
show2id = dict((sid, i) for (i, sid) in enumerate(unique_sid))
profile2id = dict((pid, i) for (i, pid) in enumerate(unique_uid))

In [ ]:
pro_dir = os.path.join(DATA_DIR, '.')

if not os.path.exists(pro_dir):
    os.makedirs(pro_dir)

with open(os.path.join(pro_dir, 'unique_sid.txt'), 'w') as f:
    for sid in unique_sid:
        f.write('%s\n' % sid)

In [ ]:
def split_train_test_proportion(data, test_prop=0.2):
    data_grouped_by_user = data.groupby('userId')
    tr_list, te_list = list(), list()
    np.random.seed(42)

    for i, (_, group) in enumerate(data_grouped_by_user):
        n_items_u = len(group)

        if n_items_u >= 5:
            idx = np.zeros(n_items_u, dtype='bool')
            idx[np.random.choice(n_items_u, size=int(test_prop * n_items_u), replace=False).astype('int64')] = True

            tr_list.append(group[np.logical_not(idx)])
            te_list.append(group[idx])
        else:
            tr_list.append(group)

        if i % 999 == 0 and i != 0:
            print(f"{i+1} users sampled")
            sys.stdout.flush()

    data_tr = pd.concat(tr_list)
    data_te = pd.concat(te_list)
    
    return data_tr, data_te

Валидационная выборка

In [ ]:
vad_plays = raw_data.loc[raw_data['userId'].isin(vd_users)]
#vad_plays = vad_plays.loc[vad_plays['movieId'].isin(unique_sid)]

#print(vad_plays)

vad_plays_tr, vad_plays_te = split_train_test_proportion(vad_plays)
vad_plays_tr.shape, vad_plays_te.shape

Тестовая выборка

In [ ]:
test_plays = raw_data.loc[raw_data['userId'].isin(te_users)]
test_plays = test_plays.loc[test_plays['movieId'].isin(unique_sid)]

test_plays_tr, test_plays_te = split_train_test_proportion(test_plays)
test_plays_tr.shape, test_plays_te.shape

vad_plays_tr.userId.nunique(), test_plays_tr.userId.nunique()

Сохраним (user_id, item_id) формат данных для модели

In [ ]:
def numerize(df, filename):
    df.userId = df.userId.map(profile2id)
    df.movieId = df.movieId.map(show2id)
    df.rename(columns={'userId': 'uid', 'movieId': 'sid'}, inplace=True)
    df.dropna(inplace=True)
    df[['uid', 'sid']].to_csv(os.path.join(pro_dir, filename), index=False)

In [ ]:
train_data = numerize(train_plays, 'train.csv')
vad_data_tr = numerize(vad_plays_tr, 'validation_tr.csv')
vad_data_te = numerize(vad_plays_te, 'validation_te.csv')
test_data_tr = numerize(test_plays_tr, 'test_tr.csv')
test_data_te = numerize(test_plays_te, 'test_te.csv')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.init import constant_, xavier_normal_
from torch.utils.data import Dataset


def xavier_normal_initialization(module):
    r"""using `xavier_normal_`_ in PyTorch to initialize the parameters in
    nn.Embedding and nn.Linear layers. For bias in nn.Linear layers,
    using constant 0 to initialize.
    .. _`xavier_normal_`:
        https://pytorch.org/docs/stable/nn.init.html?highlight=xavier_normal_#torch.nn.init.xavier_normal_
    Examples:
        >>> self.apply(xavier_normal_initialization)
    """
    if isinstance(module, nn.Embedding):
        xavier_normal_(module.weight.data)
    elif isinstance(module, nn.Linear):
        xavier_normal_(module.weight.data)
        if module.bias is not None:
            constant_(module.bias.data, 0)



class MultiVAE(torch.nn.Module):
    r"""MultiVAE is an item-based collaborative filtering model that simultaneously ranks all items for each user.
    We implement the MultiVAE model with only user dataloader.
    """

    def __init__(
        self,
        mlp_hidden_size,
        latent_dimension,
        dropout_prob,
        n_encoder_layers,
        n_users,
        n_items,
        total_anneal_steps,
        device,
        trained=False,
    ):
        super(MultiVAE, self).__init__()

        self.layers = n_encoder_layers * [mlp_hidden_size]
        self.lat_dim = latent_dimension
        self.drop_out = dropout_prob
        self.anneal_cap = 0.2
        self.total_anneal_steps = total_anneal_steps
        self.mlp_hidden_size = mlp_hidden_size
        self.n_encoder_layers = n_encoder_layers
        self.trained = trained

        self.n_users, self.n_items = n_users, n_items

        self.update = 0

        self.encode_layer_dims = [self.n_items] + self.layers + [self.lat_dim]
        self.decode_layer_dims = [int(self.lat_dim / 2)] + self.encode_layer_dims[::-1][1:]

        self.encoder = self.mlp_layers(self.encode_layer_dims)
        self.decoder = self.mlp_layers(self.decode_layer_dims)

        self.apply(xavier_normal_initialization)

    def mlp_layers(self, layer_dims):
        mlp_modules = []
        for i, (d_in, d_out) in enumerate(zip(layer_dims[:-1], layer_dims[1:])):
            mlp_modules.append(nn.Linear(d_in, d_out))
            if i != len(layer_dims[:-1]) - 1:
                mlp_modules.append(nn.Tanh())
        return nn.Sequential(*mlp_modules)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            epsilon = torch.zeros_like(std).normal_(mean=0, std=0.01)
            return mu + epsilon * std
        else:
            return mu

    def forward(self, rating_matrix): # (bs, num_items)
        if not self.trained:
            rating_matrix[rating_matrix > 0] = 1
            h = F.normalize(rating_matrix)
            h = F.dropout(h, self.drop_out, training=self.training)
            h = self.encoder(h)
            mu = h[:, : int(self.lat_dim / 2)]
            logvar = h[:, int(self.lat_dim / 2) :]
            z = self.reparameterize(mu, logvar)
            predicted = self.decoder(z)
            return predicted, mu, logvar
        else:
            batch = rating_matrix
            user_ratings = batch["tokens"].float()
            user_ratings[user_ratings > 0] = 1
            h = F.normalize(user_ratings)
            h = self.encoder(h)
            mu = h[:, : int(self.lat_dim / 2)]
            logvar = h[:, int(self.lat_dim / 2) :]
            z = self.reparameterize(mu, logvar)
            z = F.softmax(self.decoder(z), -1)
            return z, mu, logvar

    def calculate_loss(self, batch):
        rating_matrix = batch["item"].squeeze(1)
        binary_matrix = rating_matrix.detach().clone()
        binary_matrix[binary_matrix > 0] = 1

        self.update += 1
        if self.total_anneal_steps > 0:
            anneal = min(self.anneal_cap, 1.0 * self.update / self.total_anneal_steps)
        else:
            anneal = self.anneal_cap

        z, mu, logvar = self.forward(rating_matrix)

        kl_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)) * anneal

        z = F.log_softmax(z, 1)
        ce_loss = -(z * binary_matrix).sum(1).mean()

        return ce_loss + kl_loss

    def predict(self, interaction):
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]

        rating_matrix = self.get_rating_matrix(user)

        scores, _, _ = self.forward(rating_matrix)

        return scores[[user, item]]

    @torch.no_grad()
    def full_sort_predict(self, batch):
        scores, _, _ = self.forward(batch["item"].squeeze(1))
        return scores.detach().cpu().numpy()



### Загрузим выборку

In [ ]:
unique_sid = list()
with open(os.path.join(pro_dir, 'unique_sid.txt'), 'r') as f:
    for line in f:
        unique_sid.append(line.strip())

n_items = len(unique_sid)

In [ ]:
def load_train_data(csv_file):
    tp = pd.read_csv(csv_file)
    n_users = tp[f'uid'].max() + 1

    rows, cols = tp['uid'], tp['sid']
    data = sparse.csr_matrix((np.ones_like(rows),
                             (rows, cols)), dtype='float64',
                             shape=(n_users, n_items))
    return data

In [ ]:
pro_dir = '.'
train_data = load_train_data(os.path.join(pro_dir, 'train.csv'))

In [ ]:
def load_tr_te_data(csv_file_tr, csv_file_te):
    tp_tr = pd.read_csv(csv_file_tr)
    tp_te = pd.read_csv(csv_file_te)

    start_idx = min(tp_tr['uid'].min(), tp_te['uid'].min())
    end_idx = max(tp_tr['uid'].max(), tp_te['uid'].max())

    rows_tr, cols_tr = tp_tr['uid'] - start_idx, tp_tr['sid']
    rows_te, cols_te = tp_te['uid'] - start_idx, tp_te['sid']

    data_tr = sparse.csr_matrix((np.ones_like(rows_tr),
                             (rows_tr, cols_tr)), dtype='float64', shape=(end_idx - start_idx + 1, n_items))
    data_te = sparse.csr_matrix((np.ones_like(rows_te),
                             (rows_te, cols_te)), dtype='float64', shape=(end_idx - start_idx + 1, n_items))
    return data_tr, data_te

In [ ]:
vad_data_tr, vad_data_te = load_tr_te_data(os.path.join(pro_dir, 'validation_tr.csv'),
                                           os.path.join(pro_dir, 'validation_te.csv'))

Зададим гиперпараметры

In [ ]:
N = train_data.shape[0]
idxlist = range(N)

# training batch size
batch_size = 500
batches_per_epoch = int(np.ceil(float(N) / batch_size))

N_vad = vad_data_tr.shape[0]
idxlist_vad = range(N_vad)

# validation batch size (since the entire validation set might not fit into GPU memory)
batch_size_vad = 2000

# the total number of gradient updates for annealing
total_anneal_steps = 200000
# largest annealing parameter
anneal_cap = 0.2

In [ ]:
def NDCG_binary_at_k_batch(X_pred, heldout_batch, k=100):
    '''
    normalized discounted cumulative gain@k for binary relevance
    ASSUMPTIONS: all the 0's in heldout_data indicate 0 relevance
    '''
    batch_users = X_pred.shape[0]
    idx_topk_part = bn.argpartition(-X_pred, k, axis=1)
    topk_part = X_pred[np.arange(batch_users)[:, np.newaxis],
                       idx_topk_part[:, :k]]
    idx_part = np.argsort(-topk_part, axis=1)
    # X_pred[np.arange(batch_users)[:, np.newaxis], idx_topk] is the sorted
    # topk predicted score
    idx_topk = idx_topk_part[np.arange(batch_users)[:, np.newaxis], idx_part]
    tp = 1. / np.log2(np.arange(2, k + 2))

    DCG = (heldout_batch[np.arange(batch_users)[:, np.newaxis],
                         idx_topk].toarray() * tp).sum(axis=1)
    IDCG = np.array([(tp[:min(n, k)]).sum()
                     for n in heldout_batch.getnnz(axis=1)])
    return DCG / IDCG

In [ ]:
def Recall_at_k_batch(X_pred, heldout_batch, k=100):
    batch_users = X_pred.shape[0]

    idx = bn.argpartition(-X_pred, k, axis=1)
    X_pred_binary = np.zeros_like(X_pred, dtype=bool)
    X_pred_binary[np.arange(batch_users)[:, np.newaxis], idx[:, :k]] = True

    X_true_binary = (heldout_batch > 0).toarray()
    tmp = (np.logical_and(X_true_binary, X_pred_binary).sum(axis=1)).astype(
        np.float32)
    recall = tmp / np.minimum(k, X_true_binary.sum(axis=1))
    return recall

### Обучить Multi-VAE

Для набора данных установили как генеративную функцию $f_\theta(\cdot)$, так и inference модель $g_\phi(\cdot)$ как трехслойный многослойный персептрон (MLP) с симметричной архитектурой энкодера и декодера.

Пример: Пример: генерирующая функция с размерностями [200 -> 600 -> n_items] MLP, что означает, что функция инференса бует [n_items -> 600 -> 200] MLP. Таким образом, общая архитектура Multi-VAE такова: [n_items -> 600 -> 200 -> 600 -> n_items].

In [ ]:
train_data, vad_data_tr

In [ ]:
import torch
import torch.optim as optim
import numpy as np
from sklearn.metrics import ndcg_score

# Инициализация модели
model = MultiVAE(
    mlp_hidden_size=600,
    latent_dimension=200,
    dropout_prob=0.5,
    n_encoder_layers=2,
    n_users=n_users,
    n_items=n_items,
    total_anneal_steps=total_anneal_steps,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    trained=False
)

# Оптимизатор
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Функция для вычисления NDCG
def calculate_ndcg(preds, targets):
    return ndcg_score(targets, preds)

In [ ]:

n_epochs = 50
ndcgs_vad = []
# Цикл обучения
best_ndcg = -np.inf
for epoch in range(n_epochs):

    # Валидация
    model.eval()
    ndcg_dist = []
    for bnum, st_idx in enumerate(range(0, N_vad, batch_size_vad)):
        end_idx = min(st_idx + batch_size_vad, N_vad)
        X = vad_data_tr[idxlist_vad[st_idx:end_idx]]
        
        if sparse.isspmatrix(X):
            X = X.toarray()
        X = X.astype('float32')
        
        X = torch.tensor(X, dtype=torch.float32)
        pred_val = model(X)[0].detach().cpu().numpy()
        #print(pred_val.shape)
        #pred_val[:,X.nonzero()] = -100

        #print(calculate_ndcg(pred_val, vad_data_te[idxlist_vad[st_idx:end_idx]].toarray()))
        
        ndcg_dist.append(calculate_ndcg(pred_val, vad_data_te[idxlist_vad[st_idx:end_idx]].toarray()))
    
    ndcg_dist = np.array(ndcg_dist)
    ndcg_ = ndcg_dist.mean()
    ndcgs_vad.append(ndcg_)
    
    model.train()

    for bnum, st_idx in enumerate(range(0, N, batch_size)):
        
        end_idx = min(st_idx + batch_size, N)
        X = train_data[idxlist[st_idx:end_idx]]

        
        if sparse.isspmatrix(X):
            X = X.toarray()

        X = torch.tensor(X, dtype=torch.float32)

        
        optimizer.zero_grad()
        z, mu, logvar = model(X)
        loss = model.calculate_loss({'item': X})
        loss.backward()
        optimizer.step()
        


    print(f'Epoch {epoch}, NDCG: {ndcg_}')

In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(ndcgs_vad)
plt.ylabel("Validation NDCG@100")
plt.xlabel("Epochs");

обратимся снова к датасету KION: https://github.com/irsafilo/KION_DATASET

In [ ]:
items = pd.read_csv('../../items.csv')
items.head()

## демо

In [ ]:
import numpy as np
import torch

# Предположим, что у вас есть следующие данные:
# items_dataset: DataFrame с колонками ['item_id', 'title']
# show2id: словарь, отображающий item_id во внутренний id
# train_data: пользователь-товарная матрица (например, scipy.sparse.csr_matrix)
# model: обученная модель MultiVAE

def get_user_history(user_id, train_data, items_dataset, show2id):
    """
    Возвращает историю взаимодействий пользователя с названиями товаров.
    """
    # Получаем индексы товаров, с которыми взаимодействовал пользователь
    interacted_items = train_data[user_id].nonzero()[1]  # Для sparse матрицы
    
    # Преобразуем внутренние id в item_id
    id2show = {v: k for k, v in show2id.items()}
    interacted_item_ids = [id2show[item] for item in interacted_items]
    
    # Получаем названия товаров
    history = items_dataset[items_dataset['item_id'].isin(interacted_item_ids)]
    
    return history

def get_top_recommendations(user_id, model, train_data, items_dataset, show2id, top_k=10):
    """
    Возвращает топ-K рекомендаций для пользователя с названиями товаров.
    """
    # Преобразуем данные пользователя в тензор
    user_interactions = train_data[user_id].toarray().flatten()  # Для sparse матрицы
    user_interactions = torch.tensor(user_interactions, dtype=torch.float32)
    
    # Получаем предсказания от модели
    with torch.no_grad():
        model.eval()
        scores, _, _ = model(user_interactions.unsqueeze(0))  # Добавляем batch dimension
    
    # Исключаем товары, с которыми пользователь уже взаимодействовал
    interacted_items = train_data[user_id].nonzero()[1]
    scores[:,interacted_items] = -np.inf  # Исключаем их из рекомендаций
    
    # Выбираем топ-K товаров
    top_items = torch.topk(scores, k=top_k).indices.cpu().numpy()[0]
    
    # Преобразуем внутренние id в item_id
    id2show = {v: k for k, v in show2id.items()}
    print(top_items)
    top_item_ids = [id2show[item] for item in top_items]
    
    # Получаем названия товаров
    recommendations = items_dataset[items_dataset['item_id'].isin(top_item_ids)]
    return recommendations

def interactive_demo(model, train_data, items_dataset, show2id):
    """
    Интерактивная демонстрация: показывает историю и рекомендации для случайного пользователя.
    """
    # Выбираем случайного пользователя
    user_id = np.random.randint(0, train_data.shape[0])
    print(f"Случайный пользователь: {user_id}")
    
    # Показываем историю взаимодействий
    history = get_user_history(user_id, train_data, items_dataset, show2id)
    print("\nИстория взаимодействий пользователя:")
    print(history[['item_id', 'title']])
    
    # Получаем рекомендации
    recommendations = get_top_recommendations(user_id, model, train_data, items_dataset, show2id)
    print("\nТоп-10 рекомендаций:")
    print(recommendations[['item_id', 'title']])

# Запуск демонстрации
interactive_demo(model, train_data, items, show2id)

## Посмотрим на вектора айтемов

In [ ]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

def get_item_vectors(model, items_dataset, show2id):
    """
    Возвращает векторы для всех товаров из последнего слоя MultiVAE.
    """
    # Преобразуем item_id во внутренние id
    id2show = {v: k for k, v in show2id.items()}
    item_ids = items_dataset['item_id'].map(show2id).values
    
    # Создаем тензор для всех товаров
    item_indices = torch.tensor(item_ids, dtype=torch.long)
    
    item_vectors = model.decoder[-1].weight
    
    return item_vectors.detach().cpu().numpy(), item_ids

def find_similar_items(item_id, item_vectors, items_dataset, show2id, top_k=5):
    """
    Находит топ-K наиболее похожих товаров для заданного item_id.
    """

    id2show = {v: k for k, v in show2id.items()}
    # Преобразуем item_id во внутренний id
    internal_id = show2id[item_id]
    
    # Находим индекс товара в item_vectors
    #item_index = np.where(item_id == internal_id)[0][0]
    
    # Вычисляем сходство (cosine similarity)
    similarities = cosine_similarity(item_vectors[internal_id].reshape(1, -1), item_vectors).flatten()
    
    # Сортируем по сходству
    similar_indices = np.argsort(similarities)[::-1][1:top_k + 1]  # Исключаем сам товар

    # Получаем item_id и названия похожих товаров
    similar_item_ids = [id2show[i] for i in similar_indices]
    similar_items = items_dataset[items_dataset['item_id'].isin(similar_item_ids)]
    
    return similar_items

def interactive_similarity_demo(model, items_dataset, show2id, num_samples=10, top_k=5):
    """
    Интерактивная демонстрация: показывает похожие товары для случайных товаров.
    """
    # Получаем векторы для всех товаров
    item_vectors, item_ids = get_item_vectors(model, items_dataset, show2id)
    
    # Выбираем случайные товары
    sampled_items = items_dataset.sample(n=num_samples)
    
    for _, item in sampled_items.iterrows():
        item_id = item['item_id']
        title = item['title']
        genre = item['genres']
        print(f"\nТовар: {title} (item_id: {item_id}), genre: {genre}")
        
        # Находим похожие товары
        similar_items = find_similar_items(item_id, item_vectors, items_dataset, show2id, top_k)
        print("Наиболее похожие товары:")
        print(similar_items[['item_id', 'title','genres']])

# Запуск демонстрации
interactive_similarity_demo(model, items.loc[items.item_id.isin(show2id)], show2id)

## Интерактивные рекомендации

In [ ]:
def find_movie_by_title(title, items_dataset):
    """
    Находит фильм по названию и возвращает его item_id.
    """
    movie = items_dataset[items_dataset['title'].str.contains(title, case=False, na=False)]
    if not movie.empty:
        return movie.iloc[0]['item_id']
    else:
        print(f"Фильм с названием '{title}' не найден.")
        return None

def add_movie_to_user_history(user_id, item_id, train_data, show2id):
    """
    Добавляет фильм в историю взаимодействий пользователя.
    """
    if item_id not in show2id:
        print(f"Фильм с item_id {item_id} не найден в маппинге.")
        return train_data
    
    internal_id = show2id[item_id]
    train_data[user_id, internal_id] = 1  # Добавляем взаимодействие
    print(f"Фильм с item_id {item_id} добавлен в историю пользователя {user_id}.")
    return train_data

def interactive_recommendations(model, train_data, items_dataset, show2id):
    """
    Интерактивная функция: добавляет фильмы в историю пользователя и показывает обновленные рекомендации.
    """
    # Выбираем случайного пользователя
    user_id = np.random.randint(0, train_data.shape[0])
    print(f"Случайный пользователь: {user_id}")

    user_id = 0
    
    while True:
        # Показываем текущую историю пользователя
        history = get_user_history(user_id, train_data, items_dataset, show2id)
        print("\nТекущая история пользователя:")
        print(history[['item_id', 'title']])
        
        # Получаем рекомендации
        recommendations = get_top_recommendations(user_id, model, train_data, items_dataset, show2id)
        print("\nТекущие рекомендации:")
        print(recommendations[['item_id', 'title']])
        
        # Запрашиваем название фильма
        title = input("\nВведите название фильма (или 'exit' для выхода): ")
        if title.lower() == 'exit':
            break
        
        # Находим фильм по названию
        item_id = find_movie_by_title(title, items_dataset)
        if item_id is None:
            continue
        
        # Добавляем фильм в историю пользователя
        train_data = add_movie_to_user_history(user_id, item_id, train_data, show2id)
        
        # Пересчитываем рекомендации
        recommendations = get_top_recommendations(user_id, model, train_data, items_dataset, show2id)
        print("\nОбновленные рекомендации:")
        print(recommendations[['item_id', 'title']])

# Запуск интерактивной функции
interactive_recommendations(model, train_data, items, show2id)